# Lekcija 13 - Memorija agenta s Cognee znanstvenim grafovima


## Postavljanje

Ovaj bilježnik prikazuje kako izgraditi inteligentnog **pomoćnika za kodiranje** s trajnom memorijom koristeći [**Cognee**](https://www.cognee.ai/) grafikone znanja i **Microsoft Agent Framework** (MAF).

Cognee pretvara nestrukturirani tekst u strukturirani, upitni graf znanja podržan vektorskim ugradnjama — dajući vašem agentu bogatu, na odnosima svjesnu dugoročnu memoriju.

### Što ćete naučiti
1. **Izgradnja grafikona znanja**: Pretvorite profile programera i najbolje prakse u strukturirano, upitno znanje.
2. **Integracija Cognee s MAF**: Koristite `@tool` funkcije da omogućite MAF agentu da izrađuje upite na Cognee grafikonu znanja.
3. **Konverzacije svjesne sesije**: Održavajte kontekst kroz više pitanja u istoj sesiji.
4. **Dugoročna memorija**: Sačuvajte važne informacije kroz sesije i dohvatite ih u novim razgovorima.

### Preduvjeti
- Python 3.9+
- Redis koji radi lokalno (`docker run -d -p 6379:6379 redis`) za upravljanje sesijama
- API ključ za LLM (npr. OpenAI) — postavite `LLM_API_KEY` u `.env`
- `CACHING=true` u `.env` (potrebno za Cognee sesije)
- Azure AI Foundry projekt s implementiranim modelom za chat
- `AZURE_AI_PROJECT_ENDPOINT` i `AZURE_AI_MODEL_DEPLOYMENT_NAME` u `.env`
- Azure CLI autentikacija (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Vrste memorije agenta

Ovaj bilježnik istražuje iste tri vrste memorije iz glavnog bilježnika Lekcija 13, ali koristi Cognee kao pozadinsku memoriju dugog vijeka:

| Vrsta memorije | Mehanizam | Vrijek trajanja |
|---|---|---|
| **Radna** | `agent.create_session()` (MAF) | Jedan razgovor |
| **Kratkotrajna** | Cognee cache sesije (Redis) | Jedna sesija |
| **Dugotrajna** | Cognee graf znanja + vektori | Trajna |

### Memorijska arhitektura Cogneeja
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Pripremite Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Dio 1 — Izgradnja baze znanja

Unosimo tri vrste podataka kako bismo stvorili sveobuhvatnu bazu znanja za našeg asistenta za kodiranje:

1. **Profil programera** — osobna stručnost i tehnička pozadina
2. **Najbolje prakse u Pythonu** — Zen Pythona s praktičnim smjernicama
3. **Povijesni razgovori** — prošle sesije pitanja i odgovora između programera i AI asistenata


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizirajte graf znanja

Cognee može prikazati interaktivnu HTML vizualizaciju entiteta i odnosa koje je izvukao.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obogati memoriju s Memify

`memify()` analizira graf znanja i generira inteligentna pravila — identificirajući obrasce, najbolje prakse i odnose između pojmova.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Dio 2 — MAF Agent s Cognee alatima

Sada kreiramo MAF agenta koji može upitavati Cogneeov graf znanja putem `@tool` funkcija. To omogućuje agentu da iskoristi punu snagu semantičkog pretraživanja svjesnog grafova, istovremeno održavajući kontekst razgovora kroz sesije.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Radna memorija sa sesijama

`AgentSession` (kreiran putem `agent.create_session()`) pruža radnu memoriju unutar sesije. Agent se može pozvati na ranije poruke, istovremeno upitujući Cogneeov graf dugoročnog znanja.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nova sesija — Dugoročno pamćenje ostaje

Pokretanje nove sesije briše radnu memoriju, ali Cognee graf znanja i dalje je dostupan. Agent može dohvatiti isto dugoročno znanje u potpuno novom razgovoru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sažetak

U ovom bilježniku izgradili ste pomoćnika za kodiranje koji kombinira **MAF-ovu radnu memoriju** (`agent.create_session()`) s **Cogneeovim dugoročnim grafom znanja**.

### Što ste naučili
1. **Izgradnja grafa znanja**: Cognee prima nestrukturirani tekst i gradi graf + memoriju u vektorima.
2. **Obogaćivanje grafa s memify**: Izvedene činjenice i bogatiji odnosi na temelju postojećeg grafa.
3. **MAF + Cognee integracija**: `@tool` funkcije omogućuju MAF agentu da prirodno upitava Cogneeov graf.
4. **Radna memorija + dugotrajna memorija**: `AgentSession` (putem `agent.create_session()`) pruža kontekst sesije dok Cognee pruža trajno znanje.
5. **Filtrirano pretraživanje s NodeSets**: Ciljajte specifične podskupove grafa znanja (npr. samo načela).

### Ključne poruke
- **Cognee** pretvara sirovi tekst u strukturiranu, odnosno osviještenu memoriju — moćniju od ravne pohrane vektora.
- **`@tool` funkcije** čisto povezuju MAF agente i vanjske sustave znanja.
- **`AgentSession`** (putem `agent.create_session()`) održava kontekst po razgovoru odvojen od dugotrajno pohranjenog znanja.
- Isti graf znanja služi više sesija i agenata.

### Primjene u stvarnom svijetu
- **Razvojni kopiloti**: Pregled koda, analiza incidenata, asistenti za arhitekturu
- **Kopiloti za korisnike**: Agent za podršku temeljen na dokumentaciji proizvoda, često postavljanim pitanjima i CRM bilješkama
- **Interni stručni kopiloti**: Asistenti za politike, pravne ili sigurnosne razloge zasnovani na smjernicama
- **Ujednačeni slojevi podataka**: Kombiniranje strukturiranih i nestrukturiranih podataka u jedan traživi graf

### Sljedeći koraci
- Eksperimentirajte s vremenskom sviješću u Cogneeu
- Definirajte OWL ontologiju za kvalitetu grafa specifičnog za domen
- Dodajte petlje povratnih informacija korisnika za poboljšanje pretraživanja tijekom vremena
- Skalirajte na sustave s više agenata koji dijele isti sloj memorije Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj je dokument preveden pomoću AI prevoditeljske usluge [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, molimo imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati službenim i autentičnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakve nesporazume ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
